<a href="https://colab.research.google.com/github/CPTR295/Learning-Langchain/blob/main/SummarizationUsingLangchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
url = 'https://raw.githubusercontent.com/CPTR295/Learning-Langchain/refs/heads/main/Moby-Dick.txt'

response = requests.get(url)
response.raise_for_status()

with open("Moby-Dick.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

print("File downloaded successfully!")

File downloaded successfully!


In [41]:
def download_file(url,local_filename):
  response = requests.get(url)
  response.raise_for_status()

  with open(local_filename, 'wb') as f: #Dont use w because it wiriting text - not good for pdf and other files
    f.write(response.content)

  print(f"{local_filename}File downloaded successfully!")

In [42]:
url1 = 'https://raw.githubusercontent.com/CPTR295/Learning-Langchain/refs/heads/main/Paestum-Encyclopedia.txt'
url2 = 'https://raw.githubusercontent.com/CPTR295/Learning-Langchain/refs/heads/main/PaestumRevisited-StocholmsUniversitet.pdf'
url3 = 'https://raw.githubusercontent.com/CPTR295/Learning-Langchain/refs/heads/main/PaestumRevisited.pdf'
url4 = 'https://raw.githubusercontent.com/CPTR295/Learning-Langchain/refs/heads/main/Paestum-Britannica.docx'

download_file(url1,'Paestum-Encyclopedia.txt')
download_file(url1,'PaestumRevisited-StocholmsUniversitet.pdf')
download_file(url1,'PaestumRevisited.pdf')
download_file(url1,'Paestum-Britannica.docx')

Paestum-Encyclopedia.txtFile downloaded successfully!
PaestumRevisited-StocholmsUniversitet.pdfFile downloaded successfully!
PaestumRevisited.pdfFile downloaded successfully!
Paestum-Britannica.docxFile downloaded successfully!


In [3]:
with open('./Moby-Dick.txt', 'r') as file:
    moby_dick_book = file.read()

In [23]:
!pip install -q langchain_openai langchain_text_splitters langchain_community

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
from google.colab import userdata
api_key = userdata.get("OPENROUTER_API_KEY")

In [7]:
from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda,RunnableParallel


In [8]:
llm = ChatOpenAI( #Many providers provide same api interface as openai including openrouter
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

In [9]:
text_chunks_chain = (
    RunnableLambda(lambda x:
                   [
                       {
                           'chunk':text_chunk,
                       }
                       for text_chunk in TokenTextSplitter(chunk_size=10000,chunk_overlap=100).split_text(x) #Big chunk sizze as I dont have credits, hence less number of chunks so less calls
                   ]

                  )
)

In [11]:
#Map in MapReduce
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel(
        {
            'summary': summarize_chunk_chain | StrOutputParser()
        }

    )
)

In [12]:
#Reduce in MapReduce
summarize_summaries_prompt_template = """
Write a concise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_summaries_chain = (
    RunnableLambda( lambda x:
                   {
                       'summaries': '\n'.join([i['summary'] for i in x]),
                   })
    | summarize_summaries_prompt
    | llm
    | StrOutputParser()
)

In [13]:
map_reduce_chain = (
    text_chunks_chain
    | summarize_map_chain.map()
    | summarize_summaries_chain
)


In [17]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [18]:
print(summary)

**Summary of *Moby-Dick*, Chapters 1–4**

**The Call to Sea (Ch. 1: "Loomings")**
The narrator, **Ishmael**, driven by a "damp, drizzly November in [his] soul" and a suicidal inclination he sublimates into maritime labor, signs onto a whaling voyage. He views the sea as a philosophical cure for depression and a stage for the "ungraspable phantom of life," choosing the forecastle of a common sailor over the comforts of a passenger or officer.

**Arrival in New Bedford (Ch. 2: "The Carpet-Bag")**
Traveling from Manhattan to **New Bedford** (intending to sail from Nantucket), Ishmael misses his connection and is stranded on a freezing December night. Penniless, he wanders the waterfront until he finds **The Spouter-Inn**, kept by **Peter Coffin**. The inn is a dilapidated, ship-like structure adorned with whaling gear and a massive, smoke-blackened oil painting depicting a chaotic, sublime whaling scene (a whale impaling a ship on its mastheads).

**The Harpooneer Queequeg (Ch. 3: "The Sp

Summarizing across documents

In [26]:
!pip install -q wikipedia pypdf docx2txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [27]:
from langchain_community.document_loaders import WikipediaLoader
wikipedia_loader = WikipediaLoader(query='Paestum',load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [43]:
import os
import zipfile

file_path = "./Paestum-Britannica.docx"

print("File exists:", os.path.exists(file_path))

if os.path.exists(file_path):
    print("File size:", os.path.getsize(file_path), "bytes")
    print("Is valid DOCX ZIP:", zipfile.is_zipfile(file_path))

File exists: True
File size: 475 bytes
Is valid DOCX ZIP: False


In [44]:
with open("./Paestum-Britannica.docx", "rb") as f:
    content = f.read()

print(content[:500])

b'Paestum. Greek colony in Italy, south of Naples, of which a group of ruined Doric temples survives (c.530bc\xe2\x80\x93c.460 bc). This Doric Order has the most exaggerated entasis of any Antique example, and the very wide squat capitals on top of the shafts emphasize the primitive effect. The Paestum Order was much admired by Neo-Classical architects in C18 and C19, and was used where powerful effects were sought.\n\nBibliography\n\nJ. Curl (2001);\nDinsmoor (1950);\nC. Normand (1852);'


In [45]:
print(content.decode("utf-8", errors="ignore"))

Paestum. Greek colony in Italy, south of Naples, of which a group of ruined Doric temples survives (c.530bc–c.460 bc). This Doric Order has the most exaggerated entasis of any Antique example, and the very wide squat capitals on top of the shafts emphasize the primitive effect. The Paestum Order was much admired by Neo-Classical architects in C18 and C19, and was used where powerful effects were sought.

Bibliography

J. Curl (2001);
Dinsmoor (1950);
C. Normand (1852);


In [46]:
import os
import zipfile

file_path = "./PaestumRevisited.pdf"

print("Exists:", os.path.exists(file_path))
print("Size:", os.path.getsize(file_path), "bytes")

with open(file_path, "rb") as f:
    content = f.read(500)

print("First 500 bytes:", content)
print("PDF header:", content.startswith(b"%PDF-"))

Exists: True
Size: 475 bytes
First 500 bytes: b'Paestum. Greek colony in Italy, south of Naples, of which a group of ruined Doric temples survives (c.530bc\xe2\x80\x93c.460 bc). This Doric Order has the most exaggerated entasis of any Antique example, and the very wide squat capitals on top of the shafts emphasize the primitive effect. The Paestum Order was much admired by Neo-Classical architects in C18 and C19, and was used where powerful effects were sought.\n\nBibliography\n\nJ. Curl (2001);\nDinsmoor (1950);\nC. Normand (1852);'
PDF header: False


In [48]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import TextLoader

#word_loader = Docx2txtLoader('./Paestum-Britannica.docx')
#word_docs = word_loader.load()

#pdf_loader = PyPDFLoader('./PaestumRevisited.pdf')
#pdf_docs = pdf_loader.load()

txt_loader = TextLoader('./Paestum-Encyclopedia.txt')
txt_docs = txt_loader.load()

In [49]:
txt_loader = TextLoader('./PaestumRevisited.pdf')
pdf_docs = txt_loader.load()

In [50]:
txt_loader = TextLoader('./Paestum-Britannica.docx')
word_docs = txt_loader.load()

In [51]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [52]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

In [53]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful,
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [54]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary,
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)

        current_refined_summary = refine_chain.invoke(intermediate_step)

    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [55]:
full_summary = refine_summary(all_docs)
print(full_summary)

{'final_summary': '[ADDRESS], originally founded as [ADDRESS] around 600 BCE by Greek settlers from [ADDRESS], was a major ancient Greek city in [ADDRESS] on the coast of [ADDRESS]. The archaeological site is renowned for its three exceptionally well-preserved Doric temples dating from 550–450 BCE, dedicated to Hera and Athena. The city flourished as a Greek settlement for two centuries before falling to the Lucanians in 400 BCE and later becoming the Roman Latin colony of [ADDRESS] in 273 BCE. Decline followed due to shifting trade routes and malaria-inducing marshlands; the town was eventually abandoned in the early Middle Ages and rediscovered in the 18th century.\n\nThe ancient city covered approximately 120 hectares, though only 25 hectares containing the main temples and civic buildings have been excavated. The defensive walls remain largely intact, stretching 4.75 km with 24 towers and four main gates. Due to a lack of local marble, the temples feature massive columns with prono